In [0]:
-- Business Question 3: Zone-Level Mobility Patterns & Opportunities
-- Grain: Day of week + hour + zone role + taxi zone + weather condition
-- Metrics: Trip count, fare, distance, duration
-- Default period: 2026-03-01 through 2026-05-31 (PipelineConfig)
-- Environment contract: nyc_mobility.nyc_gold is the checked-in default.
-- Render exported SQL with python -m nyc_mobility.sql for another target.
-- Limitation: Weather is aligned to each trip's pickup date and hour.

WITH base AS (
    SELECT
        d.day_of_week,
        d.day_name,
        h.hour,
        h.time_of_day,
        w.temperature_band,
        w.precipitation_flag,
        t.pickup_location_id,
        t.dropoff_location_id,
        t.fare_amount,
        t.trip_distance,
        t.trip_duration_minutes
    FROM nyc_mobility.nyc_gold.fact_taxi_trip AS t
    JOIN nyc_mobility.nyc_gold.dim_date AS d
        ON t.pickup_date_key = d.date_key
    JOIN nyc_mobility.nyc_gold.dim_hour AS h
        ON t.pickup_hour_key = h.hour_key
    LEFT JOIN nyc_mobility.nyc_gold.fact_weather_hourly AS w
        ON t.pickup_date_key = w.date_key
       AND t.pickup_hour_key = w.hour_key
),

zone_activity AS (
    SELECT
        day_of_week,
        day_name,
        hour,
        time_of_day,
        temperature_band,
        precipitation_flag,
        'pickup' AS zone_role,
        pickup_location_id AS location_id,
        fare_amount,
        trip_distance,
        trip_duration_minutes
    FROM base

    UNION ALL

    SELECT
        day_of_week,
        day_name,
        hour,
        time_of_day,
        temperature_band,
        precipitation_flag,
        'dropoff' AS zone_role,
        dropoff_location_id AS location_id,
        fare_amount,
        trip_distance,
        trip_duration_minutes
    FROM base
),

aggregated AS (
    SELECT
        a.day_of_week,
        a.day_name,
        a.hour,
        a.time_of_day,
        a.temperature_band,
        a.precipitation_flag,
        a.zone_role,
        z.borough,
        z.zone,

        COUNT(*) AS trip_count,
        COUNT(a.fare_amount) AS fare_count,
        SUM(a.fare_amount) AS total_fare,
        COUNT(a.trip_distance) AS distance_count,
        SUM(a.trip_distance) AS total_distance,
        COUNT(a.trip_duration_minutes) AS duration_count,
        SUM(a.trip_duration_minutes) AS total_duration_minutes

    FROM zone_activity AS a
    JOIN nyc_mobility.nyc_gold.dim_zone AS z
        ON a.location_id = z.location_id

    GROUP BY
        a.day_of_week,
        a.day_name,
        a.hour,
        a.time_of_day,
        a.temperature_band,
        a.precipitation_flag,
        a.zone_role,
        z.borough,
        z.zone
)

SELECT
    *,
    
    ROUND(
        SUM(total_distance) OVER (PARTITION BY zone_role)
        / NULLIF(SUM(distance_count) OVER (PARTITION BY zone_role), 0),
        2
    ) AS avg_trip_distance,

    ROUND(
        SUM(total_distance) OVER (PARTITION BY zone_role, borough)
        / NULLIF(SUM(distance_count) OVER (PARTITION BY zone_role, borough), 0),
        2
    ) AS avg_trip_distance_by_borough,

    ROUND(
        SUM(total_fare) OVER (PARTITION BY zone_role, borough)
        / NULLIF(SUM(fare_count) OVER (PARTITION BY zone_role, borough), 0),
        2
    ) AS avg_fare_by_borough

FROM aggregated;